# Mouse annotator quality analysis

Compare abstar vs IgBLAST columns, annotate two problematic sequences, and summarize full mouse IgBLAST quality.


In [ ]:
from pathlib import Path
import os, json, subprocess
import pandas as pd
DATASET="ERP003950"; SAMPLE="ERR346596"
BASE = Path("/data/user/epishkin/results") if Path("/data/user/epishkin/results").exists() else Path("results")
COMPARE_DIR=BASE/DATASET/"annotator_compare"
IG_200=COMPARE_DIR/"output/igblast/ERR346596_200_igblast.tsv"
AB_200=COMPARE_DIR/"output/abstar/ERR346596/airr/ERR346596_200.tsv"
COL_OUT=COMPARE_DIR/"column_compare"; COL_OUT.mkdir(parents=True, exist_ok=True)
print(BASE, IG_200.exists(), AB_200.exists())


In [ ]:
ig=pd.read_csv(IG_200, sep="\t", dtype=str); ab=pd.read_csv(AB_200, sep="\t", dtype=str)
ig_cols=set(ig.columns); ab_cols=set(ab.columns)
def category_guess(col):
    cl=col.lower()
    if "sequence" in cl: return "sequence"
    if cl.startswith("v") or "v_" in cl: return "v_call_or_v_quality"
    if cl.startswith("d") or "d_" in cl: return "d_call_or_d_quality"
    if cl.startswith("j") or "j_" in cl: return "j_call_or_j_quality"
    if "cdr3" in cl or "junction" in cl: return "cdr3_or_junction"
    if any(x in cl for x in ["identity","evalue","score","support","productive","stop","in_frame"]): return "quality"
    return "other"
col_df=pd.DataFrame([{"column":c,"in_igblast":c in ig_cols,"in_abstar":c in ab_cols,"category_guess":category_guess(c)} for c in sorted(ig_cols|ab_cols)])
col_df.to_csv(COL_OUT/"ERR346596_abstar_vs_igblast_columns.tsv", sep="\t", index=False)
summary=[]
for col in sorted(ab_cols-ig_cols):
    s=ab[col]; non=s.notna() & (s.astype(str).str.len()>0) & (s.astype(str).str.lower()!="nan")
    summary.append({"column":col,"category_guess":category_guess(col),"nonempty_count":int(non.sum()),"nonempty_fraction":float(non.mean()),"n_unique_nonempty":int(s[non].nunique()),"example_values":" | ".join(list(s[non].astype(str).drop_duplicates().head(5)))})
summary=pd.DataFrame(summary); summary.to_csv(COL_OUT/"ERR346596_abstar_only_column_value_summary.tsv", sep="\t", index=False)
print("IgBLAST", len(ig_cols), "abstar", len(ab_cols), "abstar-only", len(ab_cols-ig_cols)); display(summary)


In [ ]:
PROB_DIR=BASE/DATASET/"problematic_sequences"; PROB_IN=PROB_DIR/"input"; PROB_OUT=PROB_DIR/"output"
PROB_IN.mkdir(parents=True, exist_ok=True); (PROB_OUT/"igblast").mkdir(parents=True, exist_ok=True); (PROB_OUT/"abstar").mkdir(parents=True, exist_ok=True)
problematic={"ko_seq_1":"GAGGTGCAGCTGTTGGAGTCTGGGGGAGGCTTGGTACAGCCTGGGGGGTCCCTGAGACTCTCCTGTGCAGCCTCTggattcacctttagcaactatGCCATGAGCTGGGTCCGCCAGGCTCCCGGGAAGGGGCTGGAGTGGGTCTCAGCTATTaccggtgggggtagaaggACATACTACGCAGACTCCGTGAAGGGCCGGTTCACCATCTCCAGAGACAATTCCAAGAACACGCTGTATCTGCAAATGAACAGCCTGAGAGCCGAGGACACGGCTgtgtacttctgtgcgacctcCCCCCGATTACGATATTTTGACTGGTTCACTCTTGGAATTCGCCGCCCCGCCATATGGGGTACTTCGATCTCTGGGGCCGTGGCACCCTGGTCACCGTCTCGAGT", "ko_seq_2":"GAGGTGCAGCTGTTGGAGTCTGGGGGAGGCTTGGTACAGCCTGGGGGGTCCCTGAGACTCTCTGTGCAGCCTCTggattctctttacggacatGCCATGAGCTGGGTCCGCAGGCTCCCGGGAGGGGCTGGAGTGGGTCTCAGCTATTacgggggggtaagaggACATACTACGCAGACTCGTGAGGGGCGGTTACATCTCAGAGACATCAGAAACCCTGTATCTGCAAGACGCTGAAACAAGAACGCTGTTACTCTGTGCGAAGAGGACTCGGATgactatcgctggcagagaccGTACTTCAGGGGGAGAATGAGAGAGATATGTGTGGGAAAATAAAGAGTCACGCGTCCGACT"}
fa=PROB_IN/"problematic_mouse_2seq.fa"
with fa.open("w") as f:
    for name,seq in problematic.items(): f.write(f">{name}\n{seq.upper()}\n")
print(fa, fa.stat().st_size)


In [ ]:
# OneQ: run IgBLAST on problematic FASTA
IGDATA=Path("/data/user/epishkin/igblast"); IGBLASTN=IGDATA/"bin/igblastn"
V=IGDATA/"internal_data/mouse/mouse_gl_V"; D=IGDATA/"internal_data/mouse/mouse_gl_D"; J=IGDATA/"internal_data/mouse/mouse_gl_J"; AUX=IGDATA/"optional_file/mouse_gl.aux"
os.environ["IGDATA"]=str(IGDATA); os.environ["PATH"]="/data/user/epishkin/igblast/bin:/data/user/epishkin/conda/envs/bcr_env/bin:"+os.environ.get("PATH",""); os.environ["LD_LIBRARY_PATH"]="/data/user/epishkin/conda/envs/bcr_env/lib:"+os.environ.get("LD_LIBRARY_PATH","")
ig_out=PROB_OUT/"igblast/problematic_mouse_2seq_igblast.tsv"; ig_log=PROB_OUT/"igblast/problematic_mouse_2seq_igblast.log"
cmd=[str(IGBLASTN),"-germline_db_V",str(V),"-germline_db_D",str(D),"-germline_db_J",str(J),"-organism","mouse","-ig_seqtype","Ig","-domain_system","imgt","-query",str(fa),"-auxiliary_data",str(AUX),"-outfmt","19","-num_threads","1","-out",str(ig_out)]
with ig_log.open("w") as log: rc=subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, text=True).returncode
print(rc, ig_out, ig_out.exists(), ig_out.stat().st_size if ig_out.exists() else None)


In [ ]:
# OneQ: run abstar on problematic FASTA
ab_project=PROB_OUT/"abstar/problematic_mouse_2seq_project2"; ab_project.mkdir(parents=True, exist_ok=True)
cmd=["abstar","run",str(fa),str(ab_project),"--germline_database","c57bl6","-o","airr","--n_processes","1","--verbose"]
r=subprocess.run(cmd, capture_output=True, text=True); print("rc", r.returncode); print(r.stdout[-2000:]); print(r.stderr[-2000:])
candidates=sorted(ab_project.rglob("*.tsv"), key=lambda p:p.stat().st_size, reverse=True); print(candidates)
ab_prob=pd.read_csv(candidates[0], sep="\t", dtype=str); canonical=PROB_OUT/"abstar/problematic_mouse_2seq.tsv"; ab_prob.to_csv(canonical, sep="\t", index=False); print(canonical, ab_prob.shape)


In [ ]:
def pick_col(df, candidates): return next((c for c in candidates if c in df.columns), None)
def get_value(row, col):
    if row is None or col is None: return None
    v=row.get(col); return None if pd.isna(v) else v
fields={"productive":["productive"],"stop_codon":["stop_codon","has_stop_codon","stop"],"v_call":["v_call"],"j_call":["j_call"],"v_identity":["v_identity","v_identity_aa","v_sequence_identity"],"j_identity":["j_identity","j_identity_aa","j_sequence_identity"],"v_evalue":["v_evalue","v_support","v_score"],"j_evalue":["j_evalue","j_support","j_score"],"cdr3_aa":["cdr3_aa","junction_aa"]}
ig_prob=pd.read_csv(PROB_OUT/"igblast/problematic_mouse_2seq_igblast.tsv", sep="\t", dtype=str); ab_prob=pd.read_csv(PROB_OUT/"abstar/problematic_mouse_2seq.tsv", sep="\t", dtype=str)
ig_id=pick_col(ig_prob,["sequence_id","seq_id"]); ab_id=pick_col(ab_prob,["sequence_id","seq_id"])
ig_map={r[ig_id]:r for _,r in ig_prob.iterrows()} if ig_id else {}; ab_map={r[ab_id]:r for _,r in ab_prob.iterrows()} if ab_id else {}
rows=[]
for sid in ["ko_seq_1","ko_seq_2"]:
    row={"sequence_id":sid,"in_igblast":sid in ig_map,"in_abstar":sid in ab_map}
    for name,cands in fields.items(): row[f"igblast_{name}"]=get_value(ig_map.get(sid),pick_col(ig_prob,cands)); row[f"abstar_{name}"]=get_value(ab_map.get(sid),pick_col(ab_prob,cands))
    rows.append(row)
cmp=pd.DataFrame(rows); cmp.to_csv(PROB_DIR/"problematic_sequence_comparison.tsv", sep="\t", index=False); display(cmp)


In [ ]:
FULL_IG_DIR=BASE/DATASET/"igblast"; QUALITY_DIR=FULL_IG_DIR/"quality"; QUALITY_DIR.mkdir(parents=True, exist_ok=True)
full_tsvs=sorted(FULL_IG_DIR.glob("*_igblast.tsv")); print("TSVs",len(full_tsvs)); [print(p.name,p.stat().st_size) for p in full_tsvs]


In [ ]:
def truthy_series(s): return s.astype(str).str.strip().str.lower().isin(["true","t","1","yes","y"])
def infer_stop_flag(df):
    if "stop_codon" in df.columns: return truthy_series(df["stop_codon"]), "stop_codon"
    for col in ["junction_aa","cdr3_aa","sequence_alignment_aa","v_sequence_alignment_aa"]:
        if col in df.columns: return df[col].astype(str).str.contains("*", regex=False, na=False), f"{col}_contains_star"
    raise ValueError("No stop_codon or AA sequence column found")
stop_rows=[]
for p in full_tsvs:
    sample=p.name.replace("_igblast.tsv",""); df=pd.read_csv(p, sep="\t", dtype=str); stop,method=infer_stop_flag(df); n=len(df); ns=int(stop.sum())
    stop_rows.append({"sample":sample,"n_records":n,"n_stop_codon":ns,"pct_stop_codon":100*ns/n if n else float("nan"),"source_file":str(p),"stop_detection_method":method})
stop_df=pd.DataFrame(stop_rows); overall={"n_records":int(stop_df["n_records"].sum()),"n_stop_codon":int(stop_df["n_stop_codon"].sum())}; overall["pct_stop_codon"]=100*overall["n_stop_codon"]/overall["n_records"] if overall["n_records"] else float("nan")
stop_df.to_csv(QUALITY_DIR/"mouse_stop_codon_summary.tsv", sep="\t", index=False); (QUALITY_DIR/"mouse_stop_codon_summary.json").write_text(json.dumps({"per_sample":stop_rows,"overall":overall}, indent=2)); print(overall); display(stop_df)


In [ ]:
AA_IDENTITY_THRESHOLD=85.0; EVALUE_BAD_THRESHOLD=1.0
V_AA_ID_COLS=["v_identity_aa","v_aa_identity","v_sequence_identity_aa"]; J_AA_ID_COLS=["j_identity_aa","j_aa_identity","j_sequence_identity_aa"]; V_EVALUE_COLS=["v_evalue","v_support"]; J_EVALUE_COLS=["j_evalue","j_support"]
def first_existing(df,names): return next((n for n in names if n in df.columns), None)
def to_float(s): return pd.to_numeric(s, errors="coerce")
def compute_bad_vj(df):
    v_aa=first_existing(df,V_AA_ID_COLS); j_aa=first_existing(df,J_AA_ID_COLS); ve=first_existing(df,V_EVALUE_COLS); je=first_existing(df,J_EVALUE_COLS); r=pd.DataFrame(index=df.index)
    r["sequence_id"]=df["sequence_id"] if "sequence_id" in df.columns else df.index.astype(str); r["v_call"]=df["v_call"] if "v_call" in df.columns else None; r["j_call"]=df["j_call"] if "j_call" in df.columns else None
    r["v_aa_identity"]=to_float(df[v_aa]) if v_aa else pd.NA; r["j_aa_identity"]=to_float(df[j_aa]) if j_aa else pd.NA; r["v_evalue"]=to_float(df[ve]) if ve else pd.NA; r["j_evalue"]=to_float(df[je]) if je else pd.NA
    r["bad_v_identity"]=(r["v_aa_identity"]<AA_IDENTITY_THRESHOLD).fillna(False) if v_aa else False; r["bad_j_identity"]=(r["j_aa_identity"]<AA_IDENTITY_THRESHOLD).fillna(False) if j_aa else False; r["bad_v_evalue"]=(r["v_evalue"]>EVALUE_BAD_THRESHOLD).fillna(False) if ve else False; r["bad_j_evalue"]=(r["j_evalue"]>EVALUE_BAD_THRESHOLD).fillna(False) if je else False
    r["bad_v"]=r["bad_v_identity"]|r["bad_v_evalue"]; r["bad_j"]=r["bad_j_identity"]|r["bad_j_evalue"]; r["bad_v_or_j"]=r["bad_v"]|r["bad_j"]; r["metric_basis"]="AA" if (v_aa or j_aa) else "NO_AA_IDENTITY_COLUMNS_AVAILABLE_EVALUE_ONLY_FALLBACK"; return r
summary_rows=[]; bad=[]
for p in full_tsvs:
    sample=p.name.replace("_igblast.tsv",""); df=pd.read_csv(p, sep="\t", dtype=str); q=compute_bad_vj(df); q.insert(0,"sample",sample); n=len(q); nb=int(q["bad_v_or_j"].sum()); summary_rows.append({"sample":sample,"n_records":n,"n_bad_v_or_j":nb,"pct_bad_v_or_j":100*nb/n if n else float("nan"),"n_bad_v":int(q["bad_v"].sum()),"n_bad_j":int(q["bad_j"].sum()),"metric_basis":q["metric_basis"].iloc[0] if len(q) else "unknown","aa_identity_threshold":AA_IDENTITY_THRESHOLD,"evalue_bad_threshold":EVALUE_BAD_THRESHOLD}); bad.append(q[q["bad_v_or_j"]])
vj_summary=pd.DataFrame(summary_rows); vj_bad=pd.concat(bad, ignore_index=True) if bad else pd.DataFrame(); vj_summary.to_csv(QUALITY_DIR/"mouse_vj_quality_summary.tsv", sep="\t", index=False); vj_bad.to_csv(QUALITY_DIR/"mouse_vj_quality_bad_reads.tsv", sep="\t", index=False); overall={"n_records":int(vj_summary["n_records"].sum()),"n_bad_v_or_j":int(vj_summary["n_bad_v_or_j"].sum()),"aa_identity_threshold":AA_IDENTITY_THRESHOLD,"evalue_bad_threshold":EVALUE_BAD_THRESHOLD,"metric_basis_values":sorted(vj_summary["metric_basis"].dropna().unique().tolist())}; overall["pct_bad_v_or_j"]=100*overall["n_bad_v_or_j"]/overall["n_records"] if overall["n_records"] else float("nan"); (QUALITY_DIR/"mouse_vj_quality_summary.json").write_text(json.dumps({"per_sample":summary_rows,"overall":overall}, indent=2)); print(overall); display(vj_summary)
